# SBA Feature Advisor Example

This notebook uses the package notebook utilities to resolve the local `featurizer_config.yaml` workspace, then reads SBA metadata and runs the advisor using Ollama.

In [3]:
import os
import pandas as pd

from featurization.notebook_utils import build_notebook_resolver, get_featurization_artifact_paths
from featurization.feature_advisor_util import FeatureAdvisorUtil, FeatureAdvisorPromptConfig

# 1) Start from the repo notebooks directory and resolve the workspace config
notebook_dir = os.getcwd()
resolver = build_notebook_resolver(notebook_dir)

print("Notebook dir:", notebook_dir)
print("Workspace root resolved from config parent:", resolver.working_dir)
print("Metadata path from config:", resolver.metadata_path)
print("Input data path from config:", resolver.featurization_input_path)

artifact_paths = get_featurization_artifact_paths(resolver)
print("Featurization artifact paths:")
for name, path in artifact_paths.items():
    print(f"- {name}: {path}")

# 2) Load SBA metadata anchored by the workspace config
metadata = pd.read_csv(resolver.metadata_path)
print("Metadata rows:", len(metadata))
print(metadata.head(3))

# 3) Build the advisor with the package prompt config
prompt_config = FeatureAdvisorPromptConfig.load_from_package()
advisor = FeatureAdvisorUtil(resolver=resolver, prompt_config=prompt_config)

print("Advisor output directory:", advisor.feature_advisor_dir)
print("Recommendation CSV path:", advisor.recommendations_csv_path)
print("Recommendation MD path:", advisor.recommendations_md_path)

# 4) Generate recommendations from Ollama using the configured model
ollama_model = resolver.ollama_model_name
print("Using Ollama model:", ollama_model)
llm_fn = advisor.llm_response_fn_from_ollama_model(ollama_model)

recommendations = advisor.recommend(
    metadata=metadata,
    model_intent="support vector machine",
    use_rules=True,
)

print("Recommendations generated by rule-based advisor")
print(recommendations.head(10))

recommendations

Notebook dir: /home/rajiv/programming/dd_parser_cleaner_migration/sba_migration/notebooks
Workspace root resolved from config parent: /home/rajiv/programming/dd_parser_cleaner_migration/sba_migration
Metadata path from config: /home/rajiv/programming/dd_parser_cleaner_migration/sba_migration/data/dd_cleaner/sba_loans_metadata_table.csv
Input data path from config: /home/rajiv/programming/dd_parser_cleaner_migration/sba_migration/data/dd_cleaner/sba_loans_user_cleaned.csv
Featurization artifact paths:
- featurized_dataset_path: /home/rajiv/programming/dd_parser_cleaner_migration/sba_migration/data/featurization/featurized_data.csv
- model_ready_dataset_path: /home/rajiv/programming/dd_parser_cleaner_migration/sba_migration/data/featurization/model_ready_numeric_data.csv
- feature_selection_knee_curve_path: /home/rajiv/programming/dd_parser_cleaner_migration/sba_migration/data/featurization/feature_selection_knee_curve.png
Metadata rows: 31
   Field Name                                  

,attribute,recommended_method,rationale
0,asofdate,Review metadata,The field does not match a recognized categori...
1,program,No encoding required,Numeric feature should be passed through the n...
2,locationid,No encoding required,Numeric feature should be passed through the n...
3,borrname,TF-IDF + TruncatedSVD,Short text or keyword-style field. Fit sparse ...
4,borrstreet,TF-IDF + TruncatedSVD,Short text or keyword-style field. Fit sparse ...
5,borrcity,TF-IDF + TruncatedSVD,Short text or keyword-style field. Fit sparse ...
6,borrstate,hierarchical_low_count_var_encoding,Hierarchical or geographic long-tail categoric...
7,borrzip,hierarchical_low_count_var_encoding,Hierarchical or geographic long-tail categoric...
8,grossapproval,No encoding required,Numeric feature should be passed through the n...
9,approvaldate,Review metadata,The field does not match a recognized categori...
